# Lecture 18: Regularization and Pipelines

Ridge and LASSO control model flexibility by shrinking coefficients. Their
tuning parameter must be selected inside cross-validation, while all imputing,
scaling, and encoding are fitted anew in each training fold. This follows
Gábor's Airbnb LASSO case with an explicit scikit-learn pipeline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

In [ ]:
airbnb = pd.read_csv("../../data/raw/airbnb_hackney_course.csv")
target = "price_eur"
numeric = ["accommodates", "bedrooms", "bathrooms", "minimum_nights", "number_of_reviews", "review_score"]
categorical = ["room_type", "property_type", "superhost", "instant_bookable", "kitchen", "wifi", "elevator"]
features = numeric + categorical
X = airbnb[features]
y = airbnb[target]

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train.shape, X_holdout.shape

## 1. Put every learned transformation inside the pipeline

In [ ]:
numeric_pipe = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_pipe = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocess = ColumnTransformer(
    [("numeric", numeric_pipe, numeric), ("categorical", categorical_pipe, categorical)]
)

In [ ]:
ridge_pipeline = Pipeline(
    [("preprocess", preprocess), ("model", Ridge())]
)
lasso_pipeline = Pipeline(
    [("preprocess", preprocess), ("model", Lasso(max_iter=20_000))]
)

## 2. Tune on training data only

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_search = GridSearchCV(
    ridge_pipeline,
    {"model__alpha": np.logspace(-2, 3, 10)},
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)

In [ ]:
lasso_search = GridSearchCV(
    lasso_pipeline,
    {"model__alpha": np.logspace(-2, 2, 10)},
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
lasso_search.fit(X_train, y_train)

In [ ]:
tuning_summary = pd.DataFrame(
    {
        "model": ["Ridge", "LASSO"],
        "best_alpha": [ridge_search.best_params_["model__alpha"], lasso_search.best_params_["model__alpha"]],
        "cv_rmse": [-ridge_search.best_score_, -lasso_search.best_score_],
    }
)
tuning_summary.round(3)

In [ ]:
ridge_path = pd.DataFrame(ridge_search.cv_results_)
lasso_path = pd.DataFrame(lasso_search.cv_results_)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ridge_path["param_model__alpha"].astype(float), -ridge_path["mean_test_score"], marker="o", label="Ridge")
ax.plot(lasso_path["param_model__alpha"].astype(float), -lasso_path["mean_test_score"], marker="o", label="LASSO")
ax.set_xscale("log")
ax.set(xlabel="Regularization strength (alpha)", ylabel="Cross-validated RMSE", title="Tune by validation performance")
ax.legend()
plt.show()

## 3. Inspect shrinkage and sparsity

In [ ]:
best_lasso = lasso_search.best_estimator_
feature_names = best_lasso.named_steps["preprocess"].get_feature_names_out()
lasso_coefficients = pd.Series(
    best_lasso.named_steps["model"].coef_, index=feature_names, name="coefficient"
)
lasso_coefficients.sort_values(key=abs, ascending=False).head(15).round(2)

In [ ]:
pd.Series(
    {
        "total coefficients": len(lasso_coefficients),
        "exactly zero": (lasso_coefficients == 0).sum(),
    }
)

## 4. Select by cross-validation, evaluate once on holdout

In [ ]:
searches = {"Ridge": ridge_search, "LASSO": lasso_search}
selected_name = min(searches, key=lambda name: -searches[name].best_score_)
selected_model = searches[selected_name].best_estimator_
holdout_prediction = selected_model.predict(X_holdout)
holdout_rmse = root_mean_squared_error(y_holdout, holdout_prediction)
print(f"Selected model: {selected_name}")
print(f"Untouched holdout RMSE: EUR {holdout_rmse:.2f}")

In [ ]:
holdout_results = pd.DataFrame({"observed": y_holdout, "predicted": holdout_prediction})
holdout_results.assign(error=lambda frame: frame["observed"] - frame["predicted"]).describe().round(1)

## Session 4 reconstruction handoff

Open `prompts/session04-validation-regularization-reconstruction.md`. Treat
the generated code as suspect until you verify the split order, the scope
of preprocessing, the tuning data, the metric sign, and the one-time use of
the holdout set.

### Optional extension

Add Elastic Net with both `alpha` and `l1_ratio` in the search grid. Explain
why the larger grid increases computation and why the holdout still must
not guide either hyperparameter.